# Watch a diffusion model change its mind

For fixed ARC-C questions (n=64, paired direct vs. reasoning prompts), we inspect per-step denoising trajectories from `GSAI-ML/LLaDA-8B-Instruct` and `Dream-org/Dream-v0-Instruct-7B` through `traj-eval`, compare commitment-step distributions, and test whether hidden states at step `k` vs. `k+10` are causal via adapter-level patching.

**Falsifier:** If patching layer-16 hidden states at step `k` vs. `k+10` produces no verdict flips or commitment-step shifts above matched-norm noise, the probed latents are not causal and we report that directly.

In [ ]:
import json
import statistics
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from traj_eval.io import load_trajectories
from traj_eval.metrics import answer_commitment_step, flip_rate, verdict_convergence_step

## 1-3. Sanity check: reproduce Nov 1 ARC-C 64-item baselines

In [ ]:
suites = [
    ('LLaDA-8B-Instruct direct', '../results/llada_instruct_arcc_smoke64_direct.jsonl'),
    ('LLaDA-8B-Instruct reasoning', '../results/llada_instruct_arcc_smoke64_reasoning.jsonl'),
    ('Dream-v0-Instruct-7B direct', '../results/dream_arcc_smoke64_direct.jsonl'),
    ('Dream-v0-Instruct-7B reasoning', '../results/dream_arcc_smoke64_reasoning.jsonl'),
]
for label, path in suites:
    trajs = load_trajectories(path)
    acc = sum(t.final_prediction == t.gold_answer for t in trajs) / len(trajs)
    convs = [s for t in trajs if (s := verdict_convergence_step(t)) is not None]
    print(f'{label:32s} n={len(trajs)} acc={acc:.4f} median_conv_step={statistics.median(convs):.1f}')

## 4-5. Figures 1 & 2: Commitment-step distributions and per-step verdict curves

In [ ]:
from IPython.display import Image, display
display(Image('../results/p1_figures/fig1_commitment_distribution.png'))
display(Image('../results/p1_figures/fig2_verdict_curves.png'))

## 6. Hidden-state patching: step k vs. k+10 (Figure 3)

In [ ]:
summary_path = Path('../results/p1_patching/p1_llada_patching_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    for c in summary['conditions']:
        print(f"{c['name']:12s} acc={c['accuracy']:.4f} flip_rate={c['verdict_flip_rate']:.4f} commit_delta={c['mean_commit_step_delta']:+.2f}")
    display(Image('../results/p1_figures/fig3_patching_effect.png'))
else:
    print('Run experiments/run_p1_patching.py first.')